#LAB - 10
#Training a CNN

In [ ]:
pip install torch torchvision matplotlib

In [ ]:
# --- Library Setup & Hardware Verification ---
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# Set manual seeds to ensure reproducibility across executions
torch.manual_seed(0)

# Configure training device: use GPU (CUDA) if available, otherwise default to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

PyTorch 2.11.0+cpu | device: cpu


# Experiment 1 — Parameters and Hyperparameters
We define a small Convolutional Neural Network (CNN) consisting of three convolutional blocks followed by two fully connected layers to explore trainable parameters versus configurations.

In [ ]:
# --- Small CNN Architecture Definition ---
class SmallCNN(nn.Module):
    """A small convolutional neural network consisting of three convolutional
    blocks (with MaxPool and ReLU activations) followed by two fully connected layers."""
    def __init__(self, num_classes=10, dropout=0.5):
        super().__init__()
        # First Conv layer: 3 input channels (RGB), 32 output filters, kernel size 3, padding 1
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        # Second Conv layer: increases feature depth from 32 to 64
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        # Third Conv layer: increases feature depth from 64 to 128
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        # Downsampling layer: pooling window 2x2 with stride 2 (halves spatial resolution)
        self.pool = nn.MaxPool2d(2, 2)
        # First Linear layer: flattened input of 128 channels * 4x4 spatial size to 256 neurons
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        # Regularization: randomly drops units during training to prevent overfitting
        self.drop = nn.Dropout(dropout)
        # Final output classification layer maps 256 hidden features to target classes
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        # Pass through Conv1 -> ReLU activation -> MaxPool (downsamples 32x32 to 16x16)
        x = self.pool(F.relu(self.conv1(x)))
        # Pass through Conv2 -> ReLU activation -> MaxPool (downsamples 16x16 to 8x8)
        x = self.pool(F.relu(self.conv2(x)))
        # Pass through Conv3 -> ReLU activation -> MaxPool (downsamples 8x8 to 4x4)
        x = self.pool(F.relu(self.conv3(x)))
        # Flatten spatial dimensions into a 1D vector (keeping batch dimension intact)
        x = torch.flatten(x, 1)
        # Pass through FC1 -> ReLU activation -> Dropout -> FC2 output
        return self.fc2(self.drop(F.relu(self.fc1(x))))

# Define the hyperparameter configuration dictionary
config = {"learning_rate": 1e-3, "batch_size": 128, "optimizer": "adam",
          "weight_decay": 1e-4, "dropout": 0.5, "epochs": 20}

# Instantiate the neural network model with the dropout rate from config
model = SmallCNN(dropout=config["dropout"])

# Display the count of trainable tensors and scalar parameters in the model
print("trainable tensors :", len(list(model.parameters())))
print("trainable scalars :", f"{sum(p.numel() for p in model.parameters()):,}")
print("hyperparameters :", len(config), "->", list(config))

trainable tensors : 10
trainable scalars : 620,362
hyperparameters : 6 -> ['learning_rate', 'batch_size', 'optimizer', 'weight_decay', 'dropout', 'epochs']


# Experiment 2 — The Learning-Rate Stability Window
We investigate how learning rates impact convergence behavior on a simple one-dimensional quadratic loss, illustrating stable, oscillating, and diverging regimes.

In [ ]:
a, w0 = 4.0, 1.0
# Stability bounds are calculated analytically: stability requires |1 - eta * a| < 1, so 0 < eta < 2/a
print("stability range : 0 < eta <", 2 / a)
# Optimal learning rate (fastest possible single-step convergence) occurs when |1 - eta * a| = 0 => eta = 1/a
print("fastest rate : eta =", 1 / a)

def gradient_descent(a, w0, eta, steps):
    """Simulates 1D gradient descent using recursive step: w_new = w - eta * grad = w * (1 - eta * a)."""
    w, history = w0, [w0]
    for _ in range(steps):
        w = w - eta * (a * w)
        history.append(w)
    return history

# Sweep through various learning rates to inspect the convergence/oscillation dynamics
for eta in [0.02, 0.2, 0.25, 0.45, 0.5, 0.6]:
    h = gradient_descent(a, w0, eta, 30)
    # Compute the analytical convergence scaling factor per step
    factor = abs(1 - eta * a)
    print(f"eta={eta:<5} |1 - eta*a| = {factor:.3f} w_30 = {h[-1]:+.3e}")

stability range : 0 < eta < 0.5
fastest rate : eta = 0.25
eta=0.02  |1 - eta*a| = 0.920 w_30 = +8.197e-02
eta=0.2   |1 - eta*a| = 0.200 w_30 = +1.074e-21
eta=0.25  |1 - eta*a| = 0.000 w_30 = +0.000e+00
eta=0.45  |1 - eta*a| = 0.800 w_30 = +1.238e-03
eta=0.5   |1 - eta*a| = 1.000 w_30 = +1.000e+00
eta=0.6   |1 - eta*a| = 1.400 w_30 = +2.420e+04


In [ ]:
# --- Convergence Speed Comparison ---
def iterations_needed(eps, a, w0, eta):
    """Calculates the theoretical number of steps t required to reach |w_t| <= eps.
    Derived from solving |1 - eta * a|^t * |w0| <= eps using logarithms."""
    factor = abs(1 - eta * a)
    if factor >= 1:
        return None # If factor is >= 1, the sequence never converges to 0
    return math.ceil(math.log(eps / w0) / math.log(factor))

# Verify the mathematical predictions against empirical 1D simulations
for eta in [0.02, 0.2]:
    predicted = iterations_needed(0.01, a, w0, eta)
    h = gradient_descent(a, w0, eta, 200)
    # Locate the first index where simulated weight falls below the threshold
    observed = next(k for k, w in enumerate(h) if abs(w) <= 0.01)
    print(f"eta={eta}: formula {predicted} iterations, simulation {observed}")

eta=0.02: formula 56 iterations, simulation 56
eta=0.2: formula 3 iterations, simulation 3


# Experiment 3 — Learning-Rate Schedules
We implement step decay, exponential decay, cosine annealing, and linear warmup schedules manually, then verify their outputs against PyTorch's native learning rate schedulers.

In [ ]:
# --- Custom Learning Rate Schedulers ---
def step_decay(epoch, eta0=0.01, gamma=0.1, s=10):
    """Applies a discrete step decay: multiplies base LR by gamma every s epochs."""
    return eta0 * gamma ** (epoch // s)

def exponential_decay(epoch, eta0=0.01, gamma=0.9):
    """Applies smooth exponential decay: scales LR by gamma every single epoch."""
    return eta0 * gamma ** epoch

def cosine_annealing(epoch, eta0=0.1, eta_min=0.0, T=100):
    """Applies cosine annealing decay: smoothly transitions LR from eta0 to eta_min over T epochs."""
    return eta_min + 0.5 * (eta0 - eta_min) * (1 + math.cos(math.pi * epoch / T))

def linear_warmup(step, eta0=0.1, warmup_steps=500):
    """Linearly scales LR from 0 up to eta0 over the first warmup_steps, then holds constant."""
    return eta0 * min(1.0, step / warmup_steps)

# Demonstrate and print values from each custom scheduler at different checkpoints
print("step decay  :", [f"{step_decay(e):.0e}" for e in (0, 10, 20, 30)])
print("exponential :", [round(exponential_decay(e), 5) for e in (0, 10, 20, 30)])
print("cosine      :", [round(cosine_annealing(e), 4) for e in (0, 25, 50, 75, 100)])
print("warmup      :", [round(linear_warmup(s), 3) for s in (0, 100, 250, 500, 800)])

step decay  : ['1e-02', '1e-03', '1e-04', '1e-05']
exponential : [0.01, 0.00349, 0.00122, 0.00042]
cosine      : [0.1, 0.0854, 0.05, 0.0146, 0.0]
warmup      : [0.0, 0.02, 0.05, 0.1, 0.1]


In [ ]:
# --- Verification with PyTorch Schedulers ---
def read_schedule(scheduler_cls, lr, epochs, checkpoints, **kwargs):
    """Constructs a dummy network and optimizer, steps through epochs, and tracks
    active learning rates at specified checkpoint epochs to match our manual calculations."""
    opt = torch.optim.SGD(nn.Linear(1, 1).parameters(), lr=lr)
    sched = scheduler_cls(opt, **kwargs)
    seen = []
    for e in range(epochs + 1):
        if e in checkpoints:
            seen.append(opt.param_groups[0]["lr"])
        opt.step()
        sched.step()
    return seen

# Capture learning rate schedules using PyTorch built-in objects
step_lr = read_schedule(torch.optim.lr_scheduler.StepLR, 0.01, 30, {0, 10, 20, 30},
                        step_size=10, gamma=0.1)
cos_lr = read_schedule(torch.optim.lr_scheduler.CosineAnnealingLR, 0.1, 100,
                        {0, 25, 50, 75, 100}, T_max=100, eta_min=0.0)

print("StepLR :", [f"{v:.0e}" for v in step_lr])
print("CosineAnnealingLR :", [round(v, 4) for v in cos_lr])

StepLR : ['1e-02', '1e-03', '1e-04', '1e-05']
CosineAnnealingLR : [0.1, 0.0854, 0.05, 0.0146, 0.0]


# Experiment 4 — Batch Size
We model mini-batch updates and observe how gradient variance drops proportional to $1/B$ when batch size $B$ scales.

In [ ]:
# --- Mini-batch Calculations & LR Scaling ---
def updates_per_epoch(N, B):
    """Computes total optimization updates per epoch for dataset size N and batch size B."""
    return math.ceil(N / B)

def scaled_lr(eta, B_old, B_new):
    """Computes updated learning rate under the Linear Scaling Rule (scaling LR proportionally to Batch Size)."""
    return eta * B_new / B_old

print("N=50,000 B=128 ->", updates_per_epoch(50_000, 128), "updates")
print("N=50,000 B=256 ->", updates_per_epoch(50_000, 256), "updates")
print("N=50,000 B=512 ->", updates_per_epoch(50_000, 512), "updates")
print("lr 0.05 at B=128 becomes", scaled_lr(0.05, 128, 256), "at B=256")
print("lr 0.05 at B=128 becomes", scaled_lr(0.05, 128, 512), "at B=512")
print("drop_last=True at B=128 would give", 50_000 // 128, "instead of 391")

N=50,000 B=128 -> 391 updates
N=50,000 B=256 -> 196 updates
N=50,000 B=512 -> 98 updates
lr 0.05 at B=128 becomes 0.1 at B=256
lr 0.05 at B=128 becomes 0.2 at B=512
drop_last=True at B=128 would give 390 instead of 391


In [ ]:
# --- Empirical Verification of Gradient Variance Scaling ---
torch.manual_seed(0)
data = torch.randn(4096, 20)
w = torch.randn(20, requires_grad=True)

def gradient_variance(B, trials=200):
    """Calculates variance of batch gradients across multiple trial updates to see scaling effects."""
    grads = []
    for _ in range(trials):
        # Sample a batch of size B from the synthetic dataset
        idx = torch.randint(0, data.shape[0], (B,))
        loss = ((data[idx] @ w) ** 2).mean()
        # Calculate explicit gradient with respect to model weight vector 'w'
        g, = torch.autograd.grad(loss, w)
        grads.append(g)
    # Return the mean variance of the gradient vector across trials
    return torch.stack(grads).var(dim=0).mean().item()

baseline = gradient_variance(8)
for B in [8, 16, 32, 64, 128]:
    v = gradient_variance(B)
    # Contrast the observed reduction in variance against the theoretically predicted 1/B speedup factor
    print(f"B={B:<4} variance {v:9.4f} measured drop {baseline / v:5.2f}x"
          f" 1/B predicts {B // 8}x")

B=8    variance   15.4724 measured drop  0.95x 1/B predicts 1x
B=16   variance    7.7227 measured drop  1.91x 1/B predicts 2x
B=32   variance    3.6622 measured drop  4.03x 1/B predicts 4x
B=64   variance    1.9980 measured drop  7.38x 1/B predicts 8x
B=128  variance    0.9500 measured drop 15.52x 1/B predicts 16x


# Experiment 5 — Grid Search and Random Search
We compare grid trial scaling with the statistically backed confidence metrics of random searches.

In [ ]:
# --- Grid Search Dimensionality Calculations ---
import functools, operator

def grid_trials(counts):
    """Calculates total combinations in a grid search by multiplying the search steps for each dimension."""
    return functools.reduce(operator.mul, counts, 1)

print("5 x 4 x 3 grid     =", grid_trials([5, 4, 3]), "trials")
print("4 x 3 x 2 x 2 grid =", grid_trials([4, 3, 2, 2]), "trials")
print("at 30 min each     =", grid_trials([4, 3, 2, 2]) * 30 / 60, "hours")

5 x 4 x 3 grid     = 60 trials
4 x 3 x 2 x 2 grid = 48 trials
at 30 min each     = 24.0 hours


In [ ]:
# --- Mathematical Formulation of Random Search Bounds ---
def probability_of_hit(p_good, n):
    """Computes probability P of finding at least one optimal configuration in 'n' random trials.
    Formula: P = 1 - (1 - p_good)^n"""
    return 1 - (1 - p_good) ** n

def trials_for_confidence(p_good, confidence):
    """Computes minimum trials 'n' to find at least one configuration with target confidence limit.
    Derived as: n >= ln(1 - confidence) / ln(1 - p_good)"""
    return math.ceil(math.log(1 - confidence) / math.log(1 - p_good))

In [ ]:
# --- Statistical Simulation of Random Search ---
def probability_of_hit(p_good, n):
    """P = 1 - (1 - p_good)^n"""
    return 1 - (1 - p_good) ** n

def trials_for_confidence(p_good, confidence):
    """n >= ln(1 - P) / ln(1 - p)"""
    return math.ceil(math.log(1 - confidence) / math.log(1 - p_good))

print("p=0.10, n=20 trials ->", round(probability_of_hit(0.10, 20), 6))
print("p=0.10, 95% needs   ->", trials_for_confidence(0.10, 0.95), "trials")
print("p=0.08, 95% needs   ->", trials_for_confidence(0.08, 0.95), "trials")

torch.manual_seed(0)
# Simulate sampling multiple configurations across iterations to count the percentage of successful hits
simulated = (torch.rand(20_000, 20) < 0.10).any(dim=1).float().mean().item()
print("simulation, p=0.10, n=20 ->", round(simulated, 4))

p=0.10, n=20 trials -> 0.878423
p=0.10, 95% needs   -> 29 trials
p=0.08, 95% needs   -> 36 trials
simulation, p=0.10, n=20 -> 0.8784


# Experiment 6 — Expected Improvement
We test an acquisition function balancing exploration versus exploitation in Bayesian optimization.

In [ ]:
# --- Bayesian Optimization Acquisition: Expected Improvement ---
standard_normal = torch.distributions.Normal(0.0, 1.0)

def expected_improvement(mu, sigma, f_best, xi=0.01):
    """Calculates Expected Improvement (EI) using closed-form Gaussian process statistics.
    EI = (f_best - mu - xi)*Phi(Z) + sigma*phi(Z) where Z is normalized improvement margin."""
    z = torch.tensor((f_best - mu - xi) / sigma)
    Phi = standard_normal.cdf(z).item()             # Cumulative distribution function
    phi = standard_normal.log_prob(z).exp().item()  # Probability density function
    ei = (f_best - mu - xi) * Phi + sigma * phi
    return ei, z.item(), Phi, phi

f_best, xi = 0.42, 0.01   # Set best validated score (loss) achieved and exploration coefficient
for name, mu, sigma in [("A", 0.35, 0.08), ("B", 0.40, 0.15)]:
    ei, z, Phi, phi = expected_improvement(mu, sigma, f_best, xi)
    print(f"{name}: mu={mu} sigma={sigma} Z={z:.4f} Phi={Phi:.4f} phi={phi:.4f}"
          f" EI = {ei:.5f}")

A: mu=0.35 sigma=0.08 Z=0.7500 Phi=0.7734 phi=0.3011 EI = 0.07049
B: mu=0.4 sigma=0.15 Z=0.0667 Phi=0.5266 phi=0.3981 EI = 0.06497


# Experiment 7 — Successive Halving
We configure budget planning and calculate budget saving metrics compared with brute force searches.

In [ ]:
# --- Successive Halving Schedules ---
def rungs(n, r0, eta):
    """Generates progressive hyperparameter search rungs.
    In each rung, we drop configurations by factor eta, and scale step-budget by eta."""
    out, n_k, r_k = [], n, r0
    while True:
        out.append((n_k, r_k))
        if n_k == 1:
            return out
        n_k, r_k = math.floor(n_k / eta), r_k * eta

# Evaluate and display saving ratios for classic configurations
for n, r0, eta in [(27, 1, 3), (81, 1, 3)]:
    schedule = rungs(n, r0, eta)
    budget = sum(n_k * r_k for n_k, r_k in schedule)   # Aggregated execution budget units
    brute_force = n * schedule[-1][1]
    print(f"n={n}, r0={r0}, eta={eta}")
    for k, (n_k, r_k) in enumerate(schedule):
        print(f" rung {k}: {n_k:>3} configs x {r_k:>2} epochs = {n_k * r_k:>4} epoch-units")
    print(f" total {budget} brute force {brute_force}"
          f" saving {brute_force / budget:.2f}x\n")

n=27, r0=1, eta=3
 rung 0:  27 configs x  1 epochs =   27 epoch-units
 rung 1:   9 configs x  3 epochs =   27 epoch-units
 rung 2:   3 configs x  9 epochs =   27 epoch-units
 rung 3:   1 configs x 27 epochs =   27 epoch-units
 total 108 brute force 729 saving 6.75x

n=81, r0=1, eta=3
 rung 0:  81 configs x  1 epochs =   81 epoch-units
 rung 1:  27 configs x  3 epochs =   81 epoch-units
 rung 2:   9 configs x  9 epochs =   81 epoch-units
 rung 3:   3 configs x 27 epochs =   81 epoch-units
 rung 4:   1 configs x 81 epochs =   81 epoch-units
 total 405 brute force 6561 saving 16.20x



# Experiment 8 — A Real Search on CIFAR-10
Now we put everything together: building datasets, defining search spaces, and executing a real hyperparameter search with Successive Halving.

In [ ]:
# --- Dataset Preparation for CIFAR-10 Run ---
from torch.utils.data import Subset, DataLoader

# Standard pre-processing: conversion to Tensor followed by standardization centering to interval [-1, 1]
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

cifar_train = torchvision.datasets.CIFAR10(root="./data", train=True,
                                           download=True, transform=transform)

# Create tiny subsets from full CIFAR-10 data to execute rapid hyperparameter search iterations
train_subset = Subset(cifar_train, range(0, 5000))  # 5,000 images for model updates
val_subset = Subset(cifar_train, range(5000, 7000)) # 2,000 images for evaluating validation performance

print(len(train_subset), "training images |", len(val_subset), "validation images")
print("the test split is not loaded at all in this experiment")

100%|██████████| 170M/170M [18:59<00:00, 150kB/s]


5000 training images | 2000 validation images
the test split is not loaded at all in this experiment


In [ ]:
# --- Define Hyperparameter Search Space ---
SEARCH_SPACE = {
    "lr": lambda r: 10 ** r.uniform(-4.0, -1.5),             # Sample learning rate uniformly on a logarithmic scale
    "weight_decay": lambda r: 10 ** r.uniform(-6.0, -3.0),   # Sample L2 regularization strength on a log scale
    "dropout": lambda r: r.uniform(0.0, 0.6),                 # Sample classification dropout rate on a linear scale
}

# Initialize specific search random state
rng = random.Random(0)
# Draw 9 independent candidate hyperparameter dictionaries from space distributions
configs = [{k: sample(rng) for k, sample in SEARCH_SPACE.items()} for _ in range(9)]

for i, c in enumerate(configs):
    print(f"cfg {i}: lr={c['lr']:.2e} wd={c['weight_decay']:.2e} p={c['dropout']:.2f}")

cfg 0: lr=1.29e-02 wd=1.88e-04 p=0.25
cfg 1: lr=4.44e-04 wd=3.42e-05 p=0.24
cfg 2: lr=9.11e-03 wd=8.13e-06 p=0.29
cfg 3: lr=2.87e-03 wd=5.30e-04 p=0.30
cfg 4: lr=5.07e-04 wd=1.85e-04 p=0.37
cfg 5: lr=4.23e-04 wd=5.36e-04 p=0.59
cfg 6: lr=1.06e-02 wd=5.09e-04 p=0.19
cfg 7: lr=6.68e-03 wd=4.97e-04 p=0.41
cfg 8: lr=1.51e-03 wd=2.00e-06 p=0.26


In [ ]:
# --- Configuration Evaluator (Training Loop) ---
def train_and_score(cfg, epochs, seed=0):
    """Trains a model from scratch with parameters 'cfg' for a budget of 'epochs' and evaluates validation loss."""
    torch.manual_seed(seed)
    net = SmallCNN(dropout=cfg["dropout"]).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=cfg["lr"],
                           weight_decay=cfg["weight_decay"])
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(train_subset, batch_size=128, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=256)

    # Model Optimization phase
    for _ in range(epochs):
        net.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            criterion(net(x), y).backward()
            opt.step()

    # Evaluation validation loss calculation phase
    net.eval()
    total_loss = seen = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            total_loss += criterion(net(x), y).item() * y.size(0)
            seen += y.size(0)
    return total_loss / seen

In [ ]:
# --- Successive Halving Implementation ---
def successive_halving(configs, r0=1, eta=3):
    """Runs successive halving algorithm over input candidate configs."""
    survivors, budget, rung = list(configs), 0, 0
    while True:
        print(f"rung {rung}: {len(survivors)} configs x {r0} epoch(s)")
        scored = []
        for c in survivors:
            loss = train_and_score(c, r0)
            budget += r0
            scored.append((loss, c))
            print(f" lr={c['lr']:.2e} wd={c['weight_decay']:.2e}"
                  f" p={c['dropout']:.2f} val loss {loss:.4f}")
        # Sort results by validation loss in ascending order (best performing models first)
        scored.sort(key=lambda pair: pair[0])
        if len(survivors) == 1:
            return scored[0], budget
        # Keep only top (1 / eta) of configurations for subsequent rung evaluation
        survivors = [c for _, c in scored[:max(1, len(survivors) // eta)]]
        r0 *= eta
        rung += 1
        print()

# Run the hyperparameter optimization process
(best_loss, best_cfg), budget = successive_halving(configs)

print(f"\nbest configuration: lr={best_cfg['lr']:.2e}"
      f" wd={best_cfg['weight_decay']:.2e} p={best_cfg['dropout']:.2f}")
print(f"validation loss {best_loss:.4f}")
print(f"budget spent {budget} epoch-units | training all nine for 9 epochs would cost 81")

rung 0: 9 configs x 1 epoch(s)
 lr=1.29e-02 wd=1.88e-04 p=0.25 val loss 6.1649
 lr=4.44e-04 wd=3.42e-05 p=0.24 val loss 1.9665
 lr=9.11e-03 wd=8.13e-06 p=0.29 val loss 2.3030
 lr=2.87e-03 wd=5.30e-04 p=0.30 val loss 1.8941
 lr=5.07e-04 wd=1.85e-04 p=0.37 val loss 1.9549
 lr=4.23e-04 wd=5.36e-04 p=0.59 val loss 2.0101
 lr=1.06e-02 wd=5.09e-04 p=0.19 val loss 2.2677
 lr=6.68e-03 wd=4.97e-04 p=0.41 val loss 2.1026
 lr=1.51e-03 wd=2.00e-06 p=0.26 val loss 1.8105

rung 1: 3 configs x 3 epoch(s)
 lr=1.51e-03 wd=2.00e-06 p=0.26 val loss 1.4796
 lr=2.87e-03 wd=5.30e-04 p=0.30 val loss 1.5479
 lr=5.07e-04 wd=1.85e-04 p=0.37 val loss 1.6091

rung 2: 1 configs x 9 epoch(s)
 lr=1.51e-03 wd=2.00e-06 p=0.26 val loss 1.2472

best configuration: lr=1.51e-03 wd=2.00e-06 p=0.26
validation loss 1.2472
budget spent 27 epoch-units | training all nine for 9 epochs would cost 81


```markdown
# Lab Exercises
```

```markdown
## Exercise 1 & 2: Learning-Rate Stability Window Sweeps
```

In [ ]:
# --- Exercise 1 ---
# Define the curvature of our quadratic bowl (a) and initial starting weight (w0)
a, w0 = 4.0, 1.0
steps = 200

# Simulate gradient descent with eta = 0.50 (on the stability limit: |1 - eta * a| = 1.0)
h_050 = gradient_descent(a, w0, 0.50, steps)

# Simulate gradient descent with eta = 0.49 (just inside stability window: |1 - eta * a| = 0.96 < 1.0)
h_049 = gradient_descent(a, w0, 0.49, steps)

print("Exercise 1:")
# At exactly 0.5, the weights oscillate infinitely between +1 and -1, never converging.
print(f"eta = 0.50 | final value after {steps} steps: {h_050[-1]:.4f} (absolute sequence: {h_050[:5]} ... {h_050[-5:]})")
# At 0.49, the sequence oscillates but successfully converges toward 0.
print(f"eta = 0.49 | final value after {steps} steps: {h_049[-1]:.4e} (absolute sequence: {h_049[:5]} ... {h_049[-5:]})")
print()

# --- Exercise 2 ---
# When we change 'a' to 100, the landscape becomes 25 times steeper.
a_new = 100.0
print("Exercise 2:")
print("For a = 100:")
# The stability limit becomes dramatically smaller: 2 / a_new = 2 / 100 = 0.02
print("Stability range: 0 < eta <", 2 / a_new)
# The optimal learning rate is now 1 / 100 = 0.01
print("Fastest convergence rate: eta =", 1 / a_new)

```markdown
## Exercise 3: Random Search Confidence vs Grid Search Trials
```

In [ ]:
# --- Exercise 3 ---
# The probability of sampling a hyperparameter configuration in the top 5% region
p_good = 0.05
# The target overall confidence level (99% probability of hitting at least one good config)
confidence = 0.99

# Calculate the minimum number of random trials required to achieve this confidence
trials_needed = trials_for_confidence(p_good, confidence)

# For a traditional grid search with 4 dimensions and 5 grid steps along each dimension
# The total number of configurations is the Cartesian product: 5^4
grid_size = grid_trials([5, 5, 5, 5])

print("Exercise 3:")
print(f"Random search trials needed for {confidence*100}% confidence with p={p_good}: {trials_needed}")
print(f"Grid search trials for 4 hyperparameters and 5 values each (5^4): {grid_size}")

```markdown
## Exercise 4: Successive Halving Budgets with Varying Reduction Factors
```

In [ ]:
# --- Exercise 4 ---
# Compare successive halving configurations using different base reduction factors (eta_SH)
n = 64  # Start with 64 initial candidate architectures
r0 = 1  # Base budget of 1 epoch per configuration

for eta_sh in [2, 4]:
    # Generate successive halving schedules dynamically using the rungs helper function
    schedule = rungs(n, r0, eta_sh)
    # Compute total sequential epoch budget used by the successive halving run
    budget = sum(n_k * r_k for n_k, r_k in schedule)
    # Brute-force approach: run all initial configurations up to the maximum rung budget
    brute_force = n * schedule[-1][1]

    print(f"eta_SH = {eta_sh}:")
    for k, (n_k, r_k) in enumerate(schedule):
        print(f"  rung {k}: {n_k:>3} configs x {r_k:>2} epochs = {n_k * r_k:>4} epoch-units")
    print(f"  Total Budget: {budget} epoch-units (brute force would be {brute_force} epoch-units)")
    print()